## Data Availability

This tutorial uses the **Human Immune** dataset, a multi-batch,
multi-donor scRNA-seq atlas of human PBMCs.

**To use real data**, download the `.h5ad` file from Google Drive
and place it in the `data/` directory:

```bash
# mkdir -p data
# Download from: https://drive.google.com/drive/folders/11PuJRaw_JC7xJhphQklac7eg9AuGHsCE
```

**For quick exploration**, the notebook falls back to a synthetic
mock dataset. Results will differ from real data but the API flow
is identical.


# Tutorial: Cell-type annotation with neighbour-evidence gating

This tutorial walks you through **annotating your own query dataset**
using Cytoneighbor. You will learn:

1. How to train a Cytoneighbor model on a reference
2. How to transfer labels to a query with optional Unknown retention
3. How to **visualize and interpret** the results
4. How to **export predictions** for your downstream analysis
5. How to **adapt** this workflow to your own data


## 1. Setup and data loading

The cell below loads the dataset if available, or generates a
synthetic mock dataset so you can follow along immediately.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if (PROJECT_ROOT / "cytoneighbor").is_dir():
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import cytoneighbor as cn

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")
print(f"Cytoneighbor version: {cn.__version__}")


In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "Human immune.h5ad"
USE_MOCK = False

if DATA_PATH.is_file():
    adata = sc.read_h5ad(DATA_PATH)
    print(f"Loaded real dataset: {adata.n_obs} cells x {adata.n_vars} genes")
else:
    USE_MOCK = True
    warnings.warn(
        f"Dataset not found at {DATA_PATH!r}. "
        "Generating synthetic mock data for demonstration. "
        "Download the real dataset for production use."
    )
    
    rng = np.random.default_rng(42)
    N_CELLS, N_GENES, N_BATCHES = 200, 600, 5
    
    batches = np.array([f"Batch_{i % (N_BATCHES-1)}" for i in range(N_CELLS)])
    batches[30:40] = "Villani"   # hold-out batch for LOBO
    
    type_list = ["CD4+ T cells", "CD8+ T cells", "NK cells", "NKT cells",
                 "B cells", "Monocytes", "Dendritic cells"]
    cell_types = np.array([type_list[i % 7] for i in range(N_CELLS)])
    
    type_ids = np.array([type_list.index(ct) for ct in cell_types])
    type_means = rng.normal(0, 1, size=(len(type_list), N_GENES))
    batch_list = sorted(set(batches))
    batch_ids = np.array([batch_list.index(b) for b in batches])
    batch_effects = rng.normal(0, 0.2, size=(len(batch_list), N_GENES))
    cell_noise = rng.normal(0, 0.5, size=(N_CELLS, N_GENES))
    X = type_means[type_ids] + batch_effects[batch_ids] + cell_noise
    X -= X.min(axis=0, keepdims=True)
    X += 0.1
    
    gene_names = [f"Gene_{j}" for j in range(N_GENES)]
    adata = sc.AnnData(X.astype(np.float32),
                       obs={"batch": batches, "final_annotation": cell_types},
                       dtype=np.float32)
    adata.var_names = gene_names
    print(f"Generated mock dataset: {adata.n_obs} cells x {adata.n_vars} genes")

print(f"Batches: {adata.obs['batch'].nunique()}")
print(f"Cell types: {adata.obs['final_annotation'].nunique()}")


## 2. Split into reference and query

We simulate a common real-world scenario: you have a well-annotated
reference dataset and an unlabelled query dataset from a different
batch or experiment.


In [ ]:
# Hold out one batch as the query
holdout_batch = "Villani"
ref = adata[adata.obs["batch"] != holdout_batch].copy()
query = adata[adata.obs["batch"] == holdout_batch].copy()
print(f"Reference: {ref.n_obs} cells  |  Query: {query.n_obs} cells")

# Check for labels present only in the query (incomplete reference)
ref_labels = set(ref.obs["final_annotation"].unique())
query_labels = set(query.obs["final_annotation"].unique())
missing = query_labels - ref_labels
print(f"Query labels missing from reference: {missing if missing else 'none'}")


## 3. Train Cytoneighbor on the reference

The `load_data` helper prepares the expression matrix and labels.
`fit` trains the SymRelEnc model and stores weights inside
`ref.uns['cytoneighbor']`.


In [ ]:
# Prepare data for training
X_ref = ref.X
if hasattr(X_ref, "toarray"):
    X_ref = X_ref.toarray()
labels_ref = ref.obs["final_annotation"].values
cn.load_data(ref, data=X_ref, labels=labels_ref)

# Train the model
if USE_MOCK:
    cn.fit(ref, epochs=2, d_model=64, depth=1, hidden=128,
           train_pairs_per_epoch=200, use_cuda=False, use_amp=False,
           synth_enable=False, validation_split=None, early_stopping_patience=None)
else:
    cn.fit(ref, epochs=30, early_stopping_patience=5)
print("Training complete. Model stored in ref.uns['cytoneighbor'].")


## 4. Annotate the query cells

`cn.annotate()` transfers labels using ranked neighbour evidence.
With `enable_unknown=True`, queries that meet a rejection rule are retained as *Unknown* for review.
The rule can defer correct calls and can miss types absent from the reference.


In [ ]:
# Transfer labels with unknown-gating
cn.annotate(
    ref,
    "final_annotation",
    query,
    enable_unknown=True,
    score_mode="logit",
)

# Quick look at predictions
print(query.obs['cytoneighbor_prediction'].value_counts())
print(f"\nUnknown rate: {(query.obs['cytoneighbor_prediction'] == 'Unknown').mean():.1%}")


## 5. Visualize the annotations

A picture is worth a thousand statistics. Let's see where the
predictions agree with the ground truth, and where Cytoneighbor
chose to say *Unknown*.


In [ ]:
import matplotlib.pyplot as plt

# --- Fix PCA for small query sets ---
sc.pp.normalize_total(query, target_sum=1e4)
sc.pp.log1p(query)

n_comps = min(30, query.n_obs - 1, query.n_vars - 1)
n_comps = max(1, n_comps)
sc.pp.pca(query, n_comps=n_comps)

n_neighbors = min(15, query.n_obs - 1)
n_neighbors = max(2, n_neighbors)
sc.pp.neighbors(query, n_neighbors=n_neighbors)
sc.tl.umap(query, init_pos='random')

# --- Panel 1: Ground truth vs Prediction ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc.pl.umap(
    query, color="final_annotation", ax=axes[0], show=False,
    title="Ground truth", legend_loc="right margin", frameon=True,
    size=100, alpha=0.8,
)

sc.pl.umap(
    query, color="cytoneighbor_prediction", ax=axes[1], show=False,
    title="Cytoneighbor prediction", legend_loc="right margin", frameon=True,
    size=100, alpha=0.8,
)

plt.tight_layout()
plt.show()

# --- Panel 2: Highlight Unknown cells ---
query.obs["is_unknown"] = pd.Categorical(
    query.obs["cytoneighbor_prediction"] == "Unknown"
).rename_categories({True: "Unknown", False: "Known"})

fig, ax = plt.subplots(figsize=(7, 5))

sc.pl.umap(
    query, color="is_unknown", ax=ax, show=False,
    title="Unknown cells (highlighted)",
    palette={"Unknown": "red", "Known": "lightgrey"},
    size=100, alpha=0.8,
)

plt.tight_layout()
plt.show()

# --- Panel 3: Confidence score distribution ---
if "cytoneighbor_confidence" in query.obs.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    
    known = query.obs[query.obs["cytoneighbor_prediction"] != "Unknown"]
    unknown = query.obs[query.obs["cytoneighbor_prediction"] == "Unknown"]
    
    ax.hist(known["cytoneighbor_confidence"], bins=30, alpha=0.7, label="Known", color="steelblue")
    if len(unknown) > 0:
        ax.hist(unknown["cytoneighbor_confidence"], bins=30, alpha=0.7, label="Unknown", color="coral")
    
    ax.set_xlabel("Confidence score")
    ax.set_ylabel("Cell count")
    ax.legend()
    ax.set_title("Prediction confidence by status")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.show()

## 6. Export results for your pipeline

Once satisfied with the annotations, export them for downstream
analysis (differential expression, trajectory inference, etc.).


In [ ]:
# Option 1: Save the full AnnData with predictions embedded
query.write("annotated_query.h5ad")
print("Saved: annotated_query.h5ad")

# Option 2: Export predictions as a CSV
export_cols = ["cytoneighbor_prediction", "cytoneighbor_confidence"]
available = [c for c in export_cols if c in query.obs.columns]
query.obs[available].to_csv("cytoneighbor_predictions.csv")
print("Saved: cytoneighbor_predictions.csv")

# Option 3: Use predictions as input to other tools
# e.g., run differential expression per predicted cell type
# sc.tl.rank_genes_groups(query, groupby="cytoneighbor_prediction")


## 7. Adapt to your own data

To use this workflow on your own data, you only need to change
three things:

```python
import cytoneighbor as cn
import scanpy as sc

# 1. Load YOUR data
ref  = sc.read_h5ad("your_reference.h5ad")
query = sc.read_h5ad("your_query.h5ad")

# 2. Specify YOUR label column
cn.load_data(ref, data=ref.X, labels=ref.obs["your_cell_type_column"].to_numpy())
cn.fit(ref)

# 3. Annotate
cn.annotate(ref, "your_cell_type_column", query, enable_unknown=True)
```

Align reference and query genes and apply consistent preprocessing. For
a query without known labels, visualize predictions and Unknown calls.


## Summary

You have learned how to:

- Train a Cytoneighbor model on a reference dataset
- Annotate a query dataset with **Unknown retention** enabled
- **Visualize** predictions on UMAP and inspect class preference scores
- **Export** results for your downstream analysis
- **Adapt** the workflow to your own data

Next steps:
- Try the [Integration tutorial](tutorial_integration.ipynb)
  to learn how Cytoneighbor handles multi-batch data
- See the [reproduction guide](../repro/) for the original experiments and the revision analysis replay
